In [ ]:
# Create Power BI semantic model for restaurant analytics
# This notebook creates a semantic model with fact/dim relationships and DAX measures

import sempy.fabric as fabric
from sempy.sempy import publish_semantic_model, create_semantic_model

# Connect to workspace
workspace = fabric.get_workspace_by_name('Fabric_IQ_Restaurant')
print(f'Connected to workspace: {workspace.display_name} ({workspace.id})')

# Get lakehouse
lakehouse = fabric.get_item_by_name('restaurant_lakehouse', item_type='Lakehouse')
print(f'Found lakehouse: {lakehouse.display_name}')

In [ ]:
# Create semantic model
model_name = 'RestaurantAnalytics'
model_description = 'Semantic model for restaurant operations intelligence'

# Step 1: Create model item
model = fabric.create_semantic_model(
    display_name=model_name,
    description=model_description,
    workspace_id=workspace.id
)

print(f'✓ Created semantic model: {model.display_name}')
print(f'  ID: {model.id}')

In [ ]:
# Define tables from Lakehouse
# The model will reference the 6 analytical tables:
# - fact_orders
# - fact_kitchen_flow
# - fact_agent_decisions
# - dim_stations
# - dim_channels
# - dim_order_status

tables_to_import = [
    'fact_orders',
    'fact_kitchen_flow',
    'fact_agent_decisions',
    'dim_stations',
    'dim_channels',
    'dim_order_status'
]

print(f'Tables to import: {len(tables_to_import)}')
for table_name in tables_to_import:
    print(f'  - {table_name}')

In [ ]:
# Add tables and create relationships
# Step 2: Define relationships (star schema)
# fact_orders -> dim_channels (via channel)
# fact_orders -> dim_order_status (via status)
# fact_kitchen_flow -> dim_stations (via station_id)
# fact_agent_decisions -> fact_orders (via order_id)

relationships = [
    {
        'fromTable': 'fact_orders',
        'fromColumn': 'channel',
        'toTable': 'dim_channels',
        'toColumn': 'channel_id',
        'type': 'many-to-one'
    },
    {
        'fromTable': 'fact_orders',
        'fromColumn': 'status',
        'toTable': 'dim_order_status',
        'toColumn': 'status_id',
        'type': 'many-to-one'
    },
    {
        'fromTable': 'fact_kitchen_flow',
        'fromColumn': 'station_id',
        'toTable': 'dim_stations',
        'toColumn': 'station_id',
        'type': 'many-to-one'
    },
    {
        'fromTable': 'fact_agent_decisions',
        'fromColumn': 'order_id',
        'toTable': 'fact_orders',
        'toColumn': 'order_id',
        'type': 'many-to-one'
    }
]

print(f'✓ Defined {len(relationships)} relationships')
for rel in relationships:
    print(f'  {rel["fromTable"]}.{rel["fromColumn"]} -> {rel["toTable"]}.{rel["toColumn"]}')

In [ ]:
# Step 3: Create DAX measures
measures = [
    {
        'name': 'PedidosTotales',
        'expression': 'COALESCE(COUNTROWS(fact_orders), 0)',
        'formatString': '#,##0',
        'description': 'Total number of orders'
    },
    {
        'name': 'PedidosAtrasados',
        'expression': 'CALCULATE([PedidosTotales], FILTER(fact_orders, fact_orders[is_delayed]=TRUE()))',
        'formatString': '#,##0',
        'description': 'Orders with delays beyond SLA'
    },
    {
        'name': 'AtrasoMedioMinutos',
        'expression': 'AVERAGE(fact_orders[delay_minutes])',
        'formatString': '0.00',
        'description': 'Average delay in minutes'
    },
    {
        'name': 'PedidosEnSLA',
        'expression': 'CALCULATE([PedidosTotales], FILTER(fact_orders, fact_orders[is_on_sla]=TRUE()))',
        'formatString': '#,##0',
        'description': 'Orders completed within SLA'
    },
    {
        'name': 'SLAPct',
        'expression': 'DIVIDE([PedidosEnSLA], [PedidosTotales], 0)',
        'formatString': '0.00%',
        'description': 'Percentage of orders on SLA'
    },
    {
        'name': 'TiempoMedioCocinaMinutos',
        'expression': 'AVERAGE(fact_kitchen_flow[processing_time_minutes])',
        'formatString': '0.00',
        'description': 'Average kitchen processing time'
    },
    {
        'name': 'ColaMediaEstacion',
        'expression': 'AVERAGE(fact_kitchen_flow[queue_length])',
        'formatString': '0.00',
        'description': 'Average queue length per station'
    },
    {
        'name': 'SaturationMediaEstacion',
        'expression': 'AVERAGE(fact_kitchen_flow[saturation_pct])',
        'formatString': '0.00%',
        'description': 'Average station saturation percentage'
    }
]

print(f'✓ Defined {len(measures)} KPI measures')
for measure in measures:
    print(f'  - {measure["name"]}')

In [ ]:
# Step 4: Summary
print('\n=== Semantic Model Definition ===' )
print(f'Model: {model_name}')
print(f'Description: {model_description}')
print(f'\nTables: {len(tables_to_import)}')
print(f'Relationships: {len(relationships)}')
print(f'Measures: {len(measures)}')

print(f'\n✓ Semantic model structure defined successfully')
print(f'✓ Ready for dashboard and ontology creation')